# DCM PhysNet training notebook

Workflow for acetone/DCM (`acodcm`):

1. **Jupyter** — inspect raw NPZ data  
2. **Bash** — `mmml fix-and-split` → train/valid/test  
3. **Jupyter** — check units + distributions  
4. **Bash** — `mmml physnet-train` (smoke, then full)  
5. **Bash** — curves + `physnet-evaluate`  
6. **Jupyter** — show plots / metrics  

**Paths:** run bash from the `acodcm` project root (`../..` relative to this notebook).  
On studix that is often `/mmhome/boittier/home/mmml_tutorial/acodcm`.

Training itself is long — prefer a terminal / Slurm for full epochs; use this notebook for analysis and short smokes.

## Useful `mmml` CLI commands

Discovery:

```bash
mmml -h
mmml commands
mmml examples
mmml <command> --help
mmml env                 # checkpoints / CHARMM paths
mmml doctor              # JAX / CHARMM / Packmol readiness
```

Data & QM:

| Command | Role |
|---|---|
| `fix-and-split` | Unit fixes + train/valid/test NPZs + `units_manifest.json` |
| `validate` | Schema-check an NPZ |
| `xml2npz` | Molpro XML → NPZ |
| `pyscf-evaluate` / `pyscf-evaluate-mp2` | Batch E/F/D (and ESP) labeling |
| `normal-mode-sample` | Sample along vibrational modes |
| `sample-diverse-xyz` | Diverse structure pick (SOAP) → NPZ |
| `compare-npz` | Reference vs model NPZ scatter plots |

PhysNet train / eval:

| Command | Role |
|---|---|
| `physnet-train` | Train EF model (`--config` YAML; CLI overrides file) |
| `physnet-evaluate` | Hold-out metrics / plots from a checkpoint |
| `extract-checkpoint-metrics` | Learning curves from Orbax `epoch-*` dirs |
| `orbax-to-json` | Export an Orbax epoch to JSON weights |
| `diagnose-lc-outliers` | Learning-curve sweep / outlier inspection |
| `train-joint` | Joint PhysNet + DCMNet |
| `active-learning` | Sample structures for re-labeling |

Handy `physnet-train` flags: `--save-config out.yaml` (dump resolved options and exit), `--restart`, `--physnet-checkpoint` (warm start), `--metrics-plot`, `--conversion` (display-only MAE scale), `--charges` / `--no-electrostatics`, `--best`, `--early-stop-patience`.

Handy `fix-and-split` flags: `--preserve-units`, `--energy-in/--energy-out`, `--force-in/--force-out`, `--dipole-in/--dipole-out`, `--atomic-ref`, `--flip-forces`, `--zscale-energies`, multiple `--efd` files to concatenate.

## 0. Jupyter — paths

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# This notebook lives in acodcm/notebooks/physnet_train/
NOTEBOOK_DIR = Path(".").resolve()
ROOT = NOTEBOOK_DIR.parents[1]  # .../acodcm
# On studix you may prefer:
# ROOT = Path("/mmhome/boittier/home/mmml_tutorial/acodcm")

RAW_NPZ = ROOT / "new-dcm-round-2-only_MP2_41950.npz"
SPLIT_DIR = ROOT / "out" / "splits" / "dcm"
CFG = NOTEBOOK_DIR / "dcm_notebook.yaml"
CKPT_DIR = ROOT / "ckpts" / "dcm-test"
ART = NOTEBOOK_DIR / "artifacts"

print("ROOT:", ROOT)
print("RAW :", RAW_NPZ, "exists=", RAW_NPZ.exists())
print("SPLIT:", SPLIT_DIR)

try:
    import jax
    print("jax devices:", jax.devices())
except Exception as e:
    print("jax not ready:", e)

## 1. Jupyter — inspect raw NPZ (before split)

In [ ]:
raw = dict(np.load(RAW_NPZ, allow_pickle=True))
print("keys:", sorted(raw.keys()))
for k, v in sorted(raw.items()):
    a = np.asarray(v)
    print(f"  {k:20s} shape={a.shape} dtype={a.dtype}")

N = np.asarray(raw["N"]).ravel()
E = np.asarray(raw["E"]).ravel()
F = np.asarray(raw["F"])
print(f"\nn={len(N)}  N unique={np.unique(N)}  "
      f"E[{E.min():.4f}, {E.max():.4f}]  |F|max={np.abs(F).max():.4f}")

## 2. Bash — `fix-and-split`

Run in a terminal from `acodcm` (or a notebook `%%bash` cell with `cd` to `ROOT`).

Default assumes PySCF-style units in → ASE training units out (eV, eV/Å, e·Å):

```bash
cd /path/to/acodcm   # e.g. ~/mmml_tutorial/acodcm

mmml fix-and-split \
  --efd ./new-dcm-round-2-only_MP2_41950.npz \
  --output-dir ./out/splits/dcm \
  --train-frac 0.8 \
  --valid-frac 0.1 \
  --test-frac 0.1 \
  --seed 42 \
  --energy-in hartree \
  --energy-out ev \
  --force-in hartree-bohr \
  --force-out ev-angstrom \
  --dipole-in debye \
  --dipole-out e-angstrom

# Optional atomic reference subtraction:
#   --atomic-ref pbe0/def2-tzvp

# Already in training units? split only:
# mmml fix-and-split --efd ./mp2_nms15_clean_train.npz \
#   --output-dir ./out/splits/mp2_nms15_resplit --preserve-units --seed 42

# Concatenate several sources:
# mmml fix-and-split --efd a.npz b.npz c.npz -o ./out/splits/combined --seed 42

ls -lh ./out/splits/dcm/
```

## 3. Jupyter — splits + `units_manifest`

In [ ]:
for p in sorted(SPLIT_DIR.glob("*")):
    print(f"{p.name:50s} {p.stat().st_size/1e6:7.2f} MB")

manifest = SPLIT_DIR / "units_manifest.json"
if manifest.exists():
    print("\nunits_manifest.json:")
    print(json.dumps(json.loads(manifest.read_text()), indent=2))
else:
    print("\n(no units_manifest.json yet — run fix-and-split first)")

In [ ]:
def load_efd(path: Path):
    return dict(np.load(path, allow_pickle=True))

train = load_efd(SPLIT_DIR / "energies_forces_dipoles_train.npz")
valid = load_efd(SPLIT_DIR / "energies_forces_dipoles_valid.npz")
test  = load_efd(SPLIT_DIR / "energies_forces_dipoles_test.npz")

for name, d in [("train", train), ("valid", valid), ("test", test)]:
    N = np.asarray(d["N"]).ravel()
    E = np.asarray(d["E"]).ravel()
    F = np.asarray(d["F"])
    print(f"{name:5s} n={len(N):6d}  N={np.unique(N)}  "
          f"E[{E.min():.3f},{E.max():.3f}]  |F|max={np.abs(F).max():.3f}")
    print("  keys:", sorted(d.keys()))

## 4. Jupyter — histograms

In [ ]:
d = train
N = np.asarray(d["N"]).ravel()
E = np.asarray(d["E"]).ravel()
F = np.asarray(d["F"])

fmag = []
for i, n in enumerate(N):
    fmag.append(np.linalg.norm(F[i, :n], axis=-1))
fmag = np.concatenate(fmag)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
ax[0].hist(E, bins=80)
ax[0].set_title("E"); ax[0].set_xlabel("energy")
ax[1].hist(fmag, bins=80)
ax[1].set_title("|F|"); ax[1].set_xlabel("force mag")

Dk = "D" if "D" in d else ("Dxyz" if "Dxyz" in d else None)
if Dk:
    D = np.asarray(d[Dk]).reshape(len(N), -1)
    ax[2].hist(np.linalg.norm(D, axis=1), bins=80)
    ax[2].set_title(f"|{Dk}|")
else:
    ax[2].text(0.5, 0.5, "no dipole", ha="center", transform=ax[2].transAxes)

fig.tight_layout()
plt.show()

## 5. Jupyter — species + one geometry

In [ ]:
if "res_name" in train:
    res = np.array([str(x) for x in train["res_name"]])
    E = np.asarray(train["E"]).ravel()
    for sp in sorted(set(res)):
        m = res == sp
        print(f"{sp:12s} n={m.sum():6d}  E[{E[m].min():.3f}, {E[m].max():.3f}]")
else:
    print("no res_name key")

In [ ]:
from ase import Atoms

i = 0
n = int(train["N"][i])
atoms = Atoms(
    numbers=np.asarray(train["Z"][i])[:n],
    positions=np.asarray(train["R"][i])[:n],
)
print(atoms)
# from ase.visualize import view; view(atoms)

## 6. Bash — write config + train

A starter YAML sits next to this notebook (`dcm_notebook.yaml`). With `valid_data` set, omit `n_train` / `n_valid` (full files are used).

```bash
cd /path/to/acodcm

# Resolve options to YAML and exit (useful sanity check):
mmml physnet-train --config notebooks/physnet_train/dcm_notebook.yaml --save-config /tmp/resolved.yaml
cat /tmp/resolved.yaml

# Smoke (~minutes)
mmml physnet-train \
  --config notebooks/physnet_train/dcm_notebook.yaml \
  --num-epochs 20 \
  --tag dcm1_smoke

# Full run (prefer terminal / Slurm)
mmml physnet-train \
  --config notebooks/physnet_train/dcm_notebook.yaml \
  --num-epochs 1200 \
  --tag dcm1
```

Equivalent long form without YAML (single-file split via `n_train`/`n_valid`):

```bash
mmml physnet-train \
  --data ./out/splits/dcm/energies_forces_dipoles_train.npz \
  --ckpt-dir ./ckpts/dcm-test \
  --tag dcm1 \
  --n-train 10000 \
  --n-valid 2000 \
  --batch-size 50 \
  --num-epochs 1200 \
  --learning-rate 0.005 \
  --optimizer adamw \
  --energy-weight 10.0 \
  --forces-weight 52.91 \
  --dipole-weight 27.21 \
  --charges-weight 14.39 \
  --charges \
  --features 32 \
  --max-degree 1 \
  --num-basis-functions 32 \
  --num-iterations 2 \
  --n-res 3 \
  --cutoff 6.0 \
  --max-atomic-number 35 \
  --zbl
```

## 7. Bash — learning curves + hold-out eval

DCM monomers are typically **10** atoms (`--natoms 10`); acetone is **20**.

```bash
cd /path/to/acodcm

RUN_DIR=$(ls -td ./ckpts/dcm-test/*/ | head -1)
echo "RUN_DIR=$RUN_DIR"

mkdir -p notebooks/physnet_train/artifacts

mmml extract-checkpoint-metrics "$RUN_DIR" \
  -o notebooks/physnet_train/artifacts/training_curves.png \
  --metrics-json notebooks/physnet_train/artifacts/training_metrics.json \
  --log-loss \
  --ef-only

mmml physnet-evaluate \
  --checkpoint "$RUN_DIR" \
  --data ./out/splits/dcm/energies_forces_dipoles_test.npz \
  --natoms 10 \
  --batch-size 50 \
  -o notebooks/physnet_train/artifacts

# Optional: export latest epoch weights
# LATEST=$(ls -d "$RUN_DIR"/epoch-* | sort -V | tail -1)
# mmml orbax-to-json "$LATEST" -o notebooks/physnet_train/artifacts/latest.json

# Optional: pred vs ref plots if evaluate wrote an NPZ
# mmml compare-npz --ref out/splits/dcm/energies_forces_dipoles_test.npz \
#   --pred notebooks/physnet_train/artifacts/...npz -o notebooks/physnet_train/artifacts/cmp
```

## 8. Jupyter — show results

In [ ]:
from IPython.display import Image, display

ART.mkdir(parents=True, exist_ok=True)
curves = ART / "training_curves.png"
metrics = ART / "metrics.json"
train_metrics = ART / "training_metrics.json"

if curves.exists():
    display(Image(filename=str(curves)))
else:
    print("missing", curves)

for p in (metrics, train_metrics):
    if p.exists():
        print(f"\n=== {p.name} ===")
        print(json.dumps(json.loads(p.read_text()), indent=2)[:4000])
    else:
        print("missing", p)

## Notes

- Always read `units_manifest.json` after `fix-and-split` before training.
- PhysNet expects **E in eV**, **F in eV/Å**, dipoles in **e·Å** (unless you deliberately train in other units and set `--conversion` only for display).
- CLI flags override YAML values in `physnet-train`.
- Sibling scripts under `acodcm/`: `08_fix_and_split_cli.sh`, `09_physnet_train_cli*.sh`, `scripts/train_lc_filtered_fresh.sh`.